# concept-graph-xai — credit-risk walkthrough

Trains a LightGBM classifier on the **Give Me Some Credit** Kaggle dataset and demonstrates every plot the package ships.

## Outline

**Part A — Setup**
1. Imports and constants
2. Load the dataset (Kaggle, with synthetic fallback)
3. Define the concept graph
4. Train LightGBM
5. Compute SHAP values

**Part B — Global concept-level analysis**
6. Sunburst by mean(|SHAP|)
7. Utilization map (sector size = feature count, colour = branch + grey for unused)
8. AUC drop per concept (permutation / SHAP-marginal / retrain)

**Part C — Concept-design diagnostics (v0.3)**
9. Block-structured feature correlation
10. Block-structured nullity correlation
11. Joint-missing-rate sunburst
12. Concept-coherence vs concept-importance scatter
13. Block-structured SHAP correlation
14. Regulatory-tag overlay

**Part D — Local explanations (v0.4)**
15. Concept violin — per-sample KDE of summed signed SHAP per concept
16. Concept waterfall — single-prediction explanation rolled up to the tree

**Part E — Static export**
17. PNG export of every figure

## Part A — Setup

### A.1 Imports and constants

In [ ]:
from __future__ import annotations

import os
from pathlib import Path

import numpy as np
import pandas as pd
import lightgbm as lgb
import shap
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import train_test_split

from concept_graph_xai import (
    ConceptGraph,
    ConceptPredictionExplainer,
    auc_drop,
    auc_drop_map,
    coherence_importance,
    coherence_importance_scatter,
    concept_violin,
    correlation_block,
    feature_correlation,
    feature_counts,
    importance_sum,
    joint_missing_map,
    joint_missing_rate,
    nullity_correlation,
    regulatory_tag_overlay,
    shap_correlation,
    sunburst,
    utilization,
    utilization_map,
)
from concept_graph_xai.adapters import from_shap_explanation

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

### A.2 Load the dataset
We try Kaggle first via `kagglehub`; if that fails we fall back to the synthetic fixture and rename columns to the GMSC schema so the rest of the notebook stays identical.

In [ ]:
GMSC_COLUMNS = [
    "RevolvingUtilizationOfUnsecuredLines",
    "age",
    "NumberOfTime30-59DaysPastDueNotWorse",
    "DebtRatio",
    "MonthlyIncome",
    "NumberOfOpenCreditLinesAndLoans",
    "NumberOfTimes90DaysLate",
    "NumberRealEstateLoansOrLines",
    "NumberOfTime60-89DaysPastDueNotWorse",
    "NumberOfDependents",
]
TARGET = "SeriousDlqin2yrs"


def load_kaggle_gmsc() -> pd.DataFrame:
    import kagglehub
    p = kagglehub.dataset_download("brycecf/give-me-some-credit-dataset")
    candidates = [f for f in os.listdir(p) if f.lower().startswith("cs-training")]
    if not candidates:
        raise FileNotFoundError(f"cs-training.csv not found in {p}")
    return pd.read_csv(os.path.join(p, candidates[0]), index_col=0)


def load_synthetic() -> pd.DataFrame:
    import sys
    repo_root = Path.cwd().parent if Path.cwd().name == "examples" else Path.cwd()
    sys.path.insert(0, str(repo_root))
    from tests.fixtures.credit_risk_toy import make_dataset
    toy = make_dataset(n=10_000, seed=RANDOM_STATE)
    rename = {
        "revolving_utilization": "RevolvingUtilizationOfUnsecuredLines",
        "n_30_59_dpd": "NumberOfTime30-59DaysPastDueNotWorse",
        "debt_ratio": "DebtRatio",
        "monthly_income": "MonthlyIncome",
        "n_90_plus_dpd": "NumberOfTimes90DaysLate",
        "n_60_89_dpd": "NumberOfTime60-89DaysPastDueNotWorse",
        "n_dependents": "NumberOfDependents",
    }
    df = toy.X.rename(columns=rename).assign(
        NumberOfOpenCreditLinesAndLoans=lambda d: np.random.poisson(8, len(d)),
        NumberRealEstateLoansOrLines=lambda d: np.random.poisson(1.0, len(d)),
    )
    df[TARGET] = toy.y
    return df.loc[:, [TARGET, *GMSC_COLUMNS]]


try:
    df = load_kaggle_gmsc()
    source = "kaggle"
except Exception as e:
    print(f"Falling back to synthetic dataset: {type(e).__name__}: {e}")
    df = load_synthetic()
    source = "synthetic"

df = df.dropna(subset=[TARGET]).copy()
for col in GMSC_COLUMNS:
    df[col] = df[col].fillna(df[col].median())

X = df[GMSC_COLUMNS]
y = df[TARGET].astype(int).to_numpy()

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=RANDOM_STATE, stratify=y
)
print(f"{source}: {len(df):,} rows, {len(GMSC_COLUMNS)} features")
print(f"train={X_train.shape}, test={X_test.shape}, prevalence={y_train.mean():.3f}")

### A.3 Define the concept graph

The graph deliberately declares `WebActivity` (under `Behaviour`) and a top-level `AlternativeData` branch even though those columns are **not** in `X`. The `utilization_map` below will render them grey — that's how you spot concepts you *meant* to model but haven't wired up yet.

In [ ]:
graph = ConceptGraph.from_dict({
    "RiskProfile": {
        "Demographics": {
            "Age": ["age"],
            "Family": ["NumberOfDependents"],
        },
        "Income": ["MonthlyIncome", "DebtRatio"],
        "Behaviour": {
            "Delinquency": [
                "NumberOfTime30-59DaysPastDueNotWorse",
                "NumberOfTime60-89DaysPastDueNotWorse",
                "NumberOfTimes90DaysLate",
            ],
            "Utilization": ["RevolvingUtilizationOfUnsecuredLines"],
            "CreditLines": [
                "NumberOfOpenCreditLinesAndLoans",
                "NumberRealEstateLoansOrLines",
            ],
            # Declared but not in X — stays grey in the utilization map.
            "WebActivity": ["web_logins_30d", "mobile_taps_30d"],
        },
        # Phantom top-level branch — declared but not modelled.
        "AlternativeData": {
            "SocialGraph": ["social_centrality"],
            "DeviceTrust": ["device_age_days"],
        },
    }
})
print(f"{len(graph.features())} features, {len(graph.concepts())} concepts")

### A.4 Train LightGBM

In [ ]:
model = lgb.LGBMClassifier(
    n_estimators=300,
    learning_rate=0.05,
    num_leaves=31,
    random_state=RANDOM_STATE,
    verbose=-1,
)
model.fit(X_train, y_train)
p_test = model.predict_proba(X_test)[:, 1]
print(f"Hold-out AUC: {roc_auc_score(y_test, p_test):.4f}")

### A.5 SHAP values

In [ ]:
explainer = shap.TreeExplainer(model)
explanation = explainer(X_test)
shap_values, feature_names = from_shap_explanation(
    explanation, feature_names=X_test.columns.tolist()
)
base_value = float(np.asarray(explainer.expected_value).reshape(-1)[-1])
print(f"SHAP shape: {shap_values.shape}, base value (logit): {base_value:+.4f}")

## Part B — Global concept-level analysis

### B.1 Sunburst by mean(|SHAP|)

In [ ]:
imp_df = importance_sum(graph, feature_names, shap_values)
# No colorscale → sunburst() colours by top-level branch (each branch gets one hue).
fig_imp = sunburst(graph, imp_df, value="importance_sum",
                   title="Concept importance (mean |SHAP|)")
fig_imp.show()

### B.2 Utilization map (sector size = feature count, colour = branch + grey for unused)
This chart now subsumes the standalone feature-count sunburst: sector area is the feature count per concept, used branches are coloured by their top-level family with hierarchical shading (sub-concepts are lighter shades), and unused branches stay grey.

In [ ]:
util_df = utilization(graph, feature_names, shap_values, threshold=0.0)
fig_util = utilization_map(graph, util_df, title="Concept utilization (grey = unused)")
fig_util.show()
util_df[["name", "kind", "used_feature_count", "feature_count", "is_used"]]

### B.3 AUC drop per concept
Three strategies, then a side-by-side table.

In [ ]:
drop_perm = auc_drop(
    graph, model, X_test, y_test,
    feature_names=X_test.columns.tolist(),
    strategy="permutation", n_repeats=10, random_state=RANDOM_STATE,
)
fig_drop_perm = auc_drop_map(graph, drop_perm, title="AUC drop — permutation")
fig_drop_perm.show()

In [ ]:
drop_shap = auc_drop(
    graph, model, X_test, y_test,
    feature_names=X_test.columns.tolist(),
    strategy="shap_marginal",
    shap_values=shap_values, base_predictions=p_test,
)
fig_drop_shap = auc_drop_map(graph, drop_shap, title="AUC drop — SHAP-marginal")
fig_drop_shap.show()

In [ ]:
def train_lgb(X, y):
    m = lgb.LGBMClassifier(n_estimators=200, learning_rate=0.05,
                           num_leaves=31, random_state=RANDOM_STATE, verbose=-1)
    m.fit(X, y)
    return m

drop_retrain = auc_drop(
    graph, model, X_test, y_test,
    feature_names=X_test.columns.tolist(),
    strategy="retrain", train_fn=train_lgb,
    X_train=X_train, y_train=y_train,
)
fig_drop_retrain = auc_drop_map(graph, drop_retrain, title="AUC drop — retrain")
fig_drop_retrain.show()

In [ ]:
compare = (
    drop_perm.loc[drop_perm["kind"] == "concept", ["name", "feature_count", "auc_drop_mean"]]
        .rename(columns={"auc_drop_mean": "permutation"})
        .merge(drop_shap.loc[drop_shap["kind"] == "concept", ["name", "auc_drop_mean"]]
                   .rename(columns={"auc_drop_mean": "shap_marginal"}), on="name")
        .merge(drop_retrain.loc[drop_retrain["kind"] == "concept", ["name", "auc_drop_mean"]]
                   .rename(columns={"auc_drop_mean": "retrain"}), on="name")
        .sort_values("permutation", ascending=False)
)
compare

## Part C — Concept-design diagnostics (v0.3)

In [ ]:
# Re-introduce missingness for the §C demo (the cleaned X has none).
X_with_missing = X_test.copy()
rng = np.random.default_rng(RANDOM_STATE)
rows = rng.choice(len(X_with_missing), size=int(len(X_with_missing) * 0.05), replace=False)
X_with_missing.loc[X_with_missing.index[rows], ["MonthlyIncome", "DebtRatio"]] = np.nan
rows2 = rng.choice(len(X_with_missing), size=int(len(X_with_missing) * 0.02), replace=False)
X_with_missing.loc[X_with_missing.index[rows2], "NumberOfDependents"] = np.nan

### C.1 Block-structured feature correlation

In [ ]:
feat_corr = feature_correlation(graph, X_test, method="spearman")
fig_feat_corr = correlation_block(feat_corr, title="Feature correlation (Spearman)")
fig_feat_corr.show()
feat_corr.block_stats.sort_values("mean_abs", ascending=False)

### C.2 Block-structured nullity correlation

In [ ]:
null_corr = nullity_correlation(graph, X_with_missing)
fig_null_corr = correlation_block(null_corr, title="Nullity correlation (Spearman)")
fig_null_corr.show()
null_corr.block_stats.sort_values("mean_abs", ascending=False)

### C.3 Joint-missing-rate sunburst

In [ ]:
jmr_df = joint_missing_rate(graph, X_with_missing)
fig_jmr = joint_missing_map(graph, jmr_df, title="Joint missing rate per concept")
fig_jmr.show()
jmr_df.loc[jmr_df["kind"] == "concept", ["name", "feature_count", "joint_missing_rate"]].sort_values("joint_missing_rate", ascending=False)

### C.4 Concept-coherence vs concept-importance scatter
The headline diagnostic. **Green** = well-designed; **red** = kitchen sink (split it); **orange** = redundant; **grey** = noise.

In [ ]:
coh_df = coherence_importance(graph, X_test, feature_names, shap_values)
fig_coh = coherence_importance_scatter(coh_df, title="Concept design diagnostic")
fig_coh.show()
coh_df.loc[coh_df["kind"] == "concept", ["name", "coherence", "importance_sum", "quadrant"]]

### C.5 Block-structured SHAP correlation

In [ ]:
shap_corr = shap_correlation(graph, feature_names, shap_values, method="spearman")
fig_shap_corr = correlation_block(shap_corr, title="SHAP correlation (Spearman)")
fig_shap_corr.show()
shap_corr.block_stats.sort_values("mean_abs", ascending=False)

### C.6 Regulatory-tag overlay

In [ ]:
import networkx as nx
g = graph.graph.copy()
tag_map = {
    "Demographics": "PII", "Age": "PII", "Family": "PII",
    "NumberOfDependents": "PII", "age": "PII",
    "Income": "financial", "MonthlyIncome": "financial", "DebtRatio": "financial",
    "Behaviour": "behavioural", "Delinquency": "behavioural",
    "Utilization": "behavioural", "CreditLines": "behavioural",
}
for node, tag in tag_map.items():
    if node in g.nodes:
        g.nodes[node]["metadata"] = {"tag": tag}
tagged_graph = ConceptGraph.from_networkx(g, root=graph.root)
fig_tag = regulatory_tag_overlay(tagged_graph, title="Concepts coloured by regulatory tag")
fig_tag.show()

## Part D — Local explanations (v0.4)
Global plots tell us what the model does on average. The plots in this section explain *individual* predictions and the per-sample distribution of effects.

### D.1 Concept violin
For each concept, a horizontal violin (KDE) of the summed signed SHAP across the held-out set. The violin width at each x is the density of samples with that contribution; a violin that bulges to the right of the dashed line consistently raises the predicted probability, one centred on zero is sometimes risk-increasing and sometimes risk-decreasing.

In [ ]:
fig_violin = concept_violin(
    graph, feature_names, shap_values,
    only_concepts=True,
    title="Concept SHAP distribution (violin)",
)
fig_violin.show()

### D.2 Concept waterfall — single-prediction explanation
`ConceptPredictionExplainer` rolls up the per-sample SHAP into the supplied tree. Pick any row in `X_test`; the waterfall starts at the SHAP base value (model's expected logit), applies each concept's contribution in descending magnitude, and ends at the predicted logit.

In [ ]:
predictor = ConceptPredictionExplainer(
    graph, model=model, X=X_test,
    shap_values=shap_values,
    base_value=base_value,
)

# Pick the highest- and lowest-risk samples to contrast
high_risk_idx = int(np.argmax(p_test))
low_risk_idx = int(np.argmin(p_test))
print(f"highest-risk row: {high_risk_idx}, P(y=1)={p_test[high_risk_idx]:.4f}")
print(f"lowest-risk  row: {low_risk_idx},  P(y=1)={p_test[low_risk_idx]:.4f}")

#### D.2a Highest-risk prediction

In [ ]:
fig_high = predictor.waterfall(high_risk_idx, depth=1,
                                title="Concept waterfall — highest-risk sample")
fig_high.show()
predictor.breakdown(high_risk_idx, depth=1)

#### D.2b Lowest-risk prediction

In [ ]:
fig_low = predictor.waterfall(low_risk_idx, depth=1,
                               title="Concept waterfall — lowest-risk sample")
fig_low.show()
predictor.breakdown(low_risk_idx, depth=1)

#### D.2c Drill into the Behaviour sub-tree (depth=2)
The same prediction at a deeper level — Behaviour splits into Delinquency, Utilization, CreditLines, and Demographics splits into Age and Family. Useful when a top-level concept is the dominant driver.

In [ ]:
fig_high_d2 = predictor.waterfall(high_risk_idx, depth=2,
                                   title="Concept waterfall (depth=2) — highest-risk sample")
fig_high_d2.show()

## Part E — Static PNG export

In [ ]:
out_dir = Path("out")
out_dir.mkdir(exist_ok=True)
exports = [
    ("importance", fig_imp),
    ("utilization", fig_util),
    ("auc_drop_permutation", fig_drop_perm),
    ("auc_drop_shap_marginal", fig_drop_shap),
    ("auc_drop_retrain", fig_drop_retrain),
    ("feature_correlation", fig_feat_corr),
    ("nullity_correlation", fig_null_corr),
    ("joint_missing", fig_jmr),
    ("coherence_importance", fig_coh),
    ("shap_correlation", fig_shap_corr),
    ("regulatory_tag", fig_tag),
    ("violin", fig_violin),
    ("waterfall_high", fig_high),
    ("waterfall_low", fig_low),
    ("waterfall_high_d2", fig_high_d2),
]
for name, fig in exports:
    try:
        fig.write_image(out_dir / f"{name}.png", width=900, height=900, scale=2)
    except Exception as e:
        print(f"PNG export skipped for {name}: {e}")
list(out_dir.iterdir())